# Statistics for Data Analysts/Engineers — Module 15: Capstone Project

This is the final module of the course. Instead of another single technique, we'll walk
through a **complete, end-to-end analysis** of one business question -- from the first
look at the data, through formulating and testing a hypothesis, controlling for
confounding variables, building a predictive model, all the way to a final report with a
recommendation. Along the way we'll use almost everything from Modules 1-14.

## Table of contents
1. [The business question and analysis plan](#sec1)
2. [Data exploration](#sec2)
3. [Testing the primary metric: conversion](#sec3)
4. [Effect size, test power, practical significance](#sec4)
5. [Side effect: did the campaign lower satisfaction?](#sec5)
6. [Attempting better targeting: a predictive model](#sec6)
7. [Final report](#sec7)
8. [Exercises and final challenge](#sec8)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. The business question and analysis plan

**Situation:** the marketing team tested a new campaign (group `B`) on a randomly
chosen half of customers, comparing it against the existing approach (group `A`).

**Business question:** *Should we roll out campaign B to ALL customers?*

To answer this rigorously, we need to answer a few supporting questions first:

1. Does campaign B genuinely increase conversion, or is the observed difference just
   sampling noise?
2. Is the effect large enough to matter for the business (not just statistically)?
3. Does the campaign have any undesirable side effects (e.g. a drop in satisfaction)?
4. Can we build a model that helps target the campaign at the customers it works best
   for?

We'll answer each of these in turn, in exactly this order.

<a id="sec2"></a>
## 2. Data exploration

Before computing any test, let's look at the data -- exactly the way we started
Module 1.

In [ ]:
print(customers.shape)
customers.info()
customers.describe()


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

conversion_rate = customers.groupby("campaign_group")["converted"].mean()
axes[0].bar(conversion_rate.index, conversion_rate.values, color=["steelblue", "darkorange"])
axes[0].set_ylabel("Conversion rate")
axes[0].set_title("Conversion by campaign group")

customers.boxplot(column="satisfaction", by="campaign_group", ax=axes[1])
axes[1].set_title("Satisfaction by campaign group")
plt.suptitle("")
plt.tight_layout()
plt.show()


The bar chart clearly suggests higher conversion in group B. The satisfaction box plot
looks fairly similar between groups at first glance -- we'll come back to that in
section 5.

<a id="sec3"></a>
## 3. Testing the primary metric: conversion

Our primary metric (decided in advance, per Module 14 section 1) is `converted`. We test
the difference in proportions with a chi-square test (Module 9) and compute a confidence
interval for the difference (Module 6):

In [ ]:
from scipy import stats

crosstab = pd.crosstab(customers["campaign_group"], customers["converted"])
chi2, p_value, df, expected = stats.chi2_contingency(crosstab, correction=False)
print(f"chi2 = {chi2:.3f}, p = {p_value:.4f}")


In [ ]:
summary = customers.groupby("campaign_group")["converted"].agg(["sum", "count", "mean"])
print(summary)

p_a, n_a = summary.loc["A", "mean"], summary.loc["A", "count"]
p_b, n_b = summary.loc["B", "mean"], summary.loc["B", "count"]

difference = p_b - p_a
se_difference = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
ci_low = difference - 1.96 * se_difference
ci_high = difference + 1.96 * se_difference

print(f"Conversion difference (B - A): {difference:.4f}")
print(f"95% CI for the difference: ({ci_low:.4f}, {ci_high:.4f})")


The p-value is clearly below `0.05`, and the `95%` confidence interval for the
difference does NOT contain zero -- both point to the same conclusion: **the conversion
difference between groups is statistically significant**. We can reject H0 (no
difference).

<a id="sec4"></a>
## 4. Effect size, test power, practical significance

Statistical significance is only half the answer (Modules 7, 11, 14). Let's check the
effect size (odds ratio, Module 13) and whether our test even had a realistic chance of
detecting an effect of this size (power, Module 14).

In [ ]:
from statsmodels.formula.api import logit
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

logistic_model = logit("converted ~ C(campaign_group)", data=customers).fit(disp=0)
odds_ratio = np.exp(logistic_model.params["C(campaign_group)[T.B]"])
print(f"Odds ratio (B vs A): {odds_ratio:.2f}")

effect_size = proportion_effectsize(p_b, p_a)
power = NormalIndPower().solve_power(
    effect_size=effect_size, nobs1=n_a, alpha=0.05, ratio=n_b / n_a, alternative="two-sided"
)
print(f"Achieved power: {power:.3f}")


**Numerical summary:** the conversion rate rises from about `12%` to about `22%`
(nearly double), the difference is statistically significant (`chi-square test,
p < 0.01`), and our test had solid power (`> 0.8`) to detect an effect of this size. This
isn't an effect that's "significant only because the sample is huge" (Module 14, section
5) -- it's a large, practically meaningful difference.

<a id="sec5"></a>
## 5. Side effect: did the campaign lower satisfaction?

Before recommending a rollout, it's worth checking whether campaign B has any
undesirable side effects. Let's start with the simplest, NAIVE comparison of
satisfaction between groups:

In [ ]:
from statsmodels.formula.api import ols

naive_model = ols("satisfaction ~ C(campaign_group)", data=customers).fit()
print(naive_model.summary().tables[1])


> ⚠️ **A worrying signal -- but not so fast**
>
> This simple model suggests group B has SIGNIFICANTLY lower satisfaction (`p` below `0.05`)! If we stopped here, we might wrongly conclude that campaign B improves conversion at the cost of customer happiness. But remember Module 11: correlation (here: a difference in means) doesn't mean causation if there's a HIDDEN VARIABLE in play. Let's check whether the two groups are actually comparable on other traits.

In [ ]:
print(customers.groupby("campaign_group")["monthly_spend"].mean())


Despite random assignment to groups, pure chance meant that in this particular sample,
group B happens to have somewhat LOWER average monthly spend than group A -- and we know
from Module 11/12 that spend correlates strongly and positively with satisfaction. This
could be exactly the hidden variable distorting the naive comparison. Let's control for
it with linear regression (Module 12):

In [ ]:
controlled_model = ols("satisfaction ~ C(campaign_group) + monthly_spend", data=customers).fit()
print(controlled_model.summary().tables[1])


Once we control for spend, the coefficient on `campaign_group[T.B]` stops being
statistically significant (`p` above `0.05`). **Conclusion: there's no reliable evidence
that campaign B hurts satisfaction** -- the apparent effect in the naive comparison was an
artifact of a chance spend imbalance between groups, not a real effect of the
campaign.

> 📊 **Always check group comparability, even with random assignment**
>
> Random allocation guarantees groups are comparable "in expectation" (on average, over infinitely many repetitions), but in any ONE particular sample, a chance imbalance in some trait can always occur -- especially with a moderate sample size. Good practice: compare key control variables between groups and, if they differ noticeably, include them as control predictors in the model.

<a id="sec6"></a>
## 6. Attempting better targeting: a predictive model

Since campaign B works, can we identify WHICH customers respond to it best, so we could
target it more precisely? Let's build a fuller logistic regression model (Module 13)
with additional predictors:

In [ ]:
full_model = logit(
    "converted ~ C(campaign_group) + C(region) + monthly_spend + age", data=customers
).fit(disp=0)
print(full_model.summary())


> ⚠️ **One seemingly significant region -- should we trust it?**
>
> Notice that one of the regions may come out with a p-value below `0.05`, while the others don't. Before building a targeting strategy around region, ask yourself the questions from Modules 7 and 10: (1) is there any PLAUSIBLE MECHANISM by which region itself would affect the effectiveness of a randomly assigned campaign? In our data, region was assigned independently of campaign group, so there's no causal reason for this. (2) here we're checking the effect of FOUR region categories at once -- this is exactly the multiple comparisons problem (Module 10): even when region has NO true effect at all, the chance that AT LEAST ONE of the four categories comes out "significant" from pure noise is much higher than 5%. The most likely explanation: this is a false alarm (Type I error), not a real effect.

In [ ]:
full_odds_ratios = np.exp(full_model.params)
print(full_odds_ratios)
print(f"\np-value for campaign group in the full model: {full_model.pvalues['C(campaign_group)[T.B]']:.4f}")


The key observation: the odds ratio for `campaign_group[T.B]` stays practically
unchanged and significant regardless of whether we control for region, spend, and age --
the campaign effect is **robust** to adding other predictors. None of the other
predictors give a consistent, reliable basis for more precise targeting in this data --
the good news is that the campaign works roughly EQUALLY well across all customer
segments, so rolling it out to the whole customer base is reasonable.

<a id="sec7"></a>
## 7. Final report

**Question:** Should we roll out campaign B to all customers?

**Recommendation: YES**, based on the following findings:

1. Campaign B raises the conversion rate from about `12%` to about `22%` -- the
   difference is statistically significant (`chi-square test, p < 0.01`) and practically
   meaningful (nearly a two-fold increase, odds ratio about `2`).
2. The test had sufficient statistical power (`> 0.8`), so this result isn't random noise
   from an underpowered sample.
3. A naive analysis suggested a drop in satisfaction in group B, but after controlling
   for a chance spend imbalance between groups, the effect disappeared -- **there is no
   reliable evidence of a negative side effect**.
4. We found no reliable basis for more precise targeting (the seemingly significant
   region effect is most likely a false alarm) -- the campaign effect appears consistent
   across all segments, which argues for a full rollout rather than restricting it to a
   subgroup of customers.

**Caveats:** it's worth monitoring conversion and satisfaction after the full rollout too
(not just during the test period), and considering a longer observation window for
satisfaction to rule out delayed effects.

<a id="sec8"></a>
## 8. Exercises and final challenge

The final exercises of this course are more open-ended than before -- this is your
chance to carry out a piece of analysis on your own, start to finish.

> 📝 **Exercise 1: Analysis by city instead of region**
>
> Repeat the analysis from sections 2-3 (exploration + chi-square test), but checking whether the conversion rate differs between `city` (5 categories) instead of `campaign_group`. Interpret the result in light of how we built this data.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
city_table = pd.crosstab(customers["city"], customers["converted"])
chi2_c, p_c, df_c, expected_c = stats.chi2_contingency(city_table, correction=False)
print(f"chi2={chi2_c:.3f}, p={p_c:.4f}")
```

In the data, conversion depends ONLY on campaign_group, not on city -- so expect no significance (p above 0.05). If it happens to come out close to the threshold, that's a good moment to recall the false-alarm discussion from section 6.
</details>

> 📝 **Exercise 2: Confidence interval for the odds ratio**
>
> Using `logistic_model` from section 4, print the `95%` confidence interval for the `campaign_group[T.B]` coefficient (the `.conf_int()` method), then transform both bounds with `np.exp` to get a confidence interval for the ODDS RATIO.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
logit_ci = logistic_model.conf_int().loc["C(campaign_group)[T.B]"]
odds_ratio_ci = np.exp(logit_ci)
print(f"95% CI for the odds ratio: ({odds_ratio_ci[0]:.2f}, {odds_ratio_ci[1]:.2f})")
```
</details>

> 📝 **Exercise 3: Your own mini-report**
>
> Pick ANY other pair of variables from `customers` (e.g. `age` and `satisfaction`, or `region` and `monthly_spend`) and write a 3-4 sentence summary in a markdown cell: what hypothesis would you propose, what test/model would you use, and what result would you expect based on what you know about how this data was built from earlier modules.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
# This exercise is open-ended -- there's no single "correct" solution.
# Example answer: "Hypothesis: age does not affect monthly_spend (both variables
# are generated independently in the data). Test: Pearson correlation (Module 11)
# or simple linear regression (Module 12). Expected result: r close to 0, no
# statistical significance."
print("See the comment in the code above.")
```
</details>

> 🔥 **Final challenge: a complete report for satisfaction as the metric**
>
> Carry out an analogous, complete end-to-end analysis to this module's, but with `satisfaction` as the PRIMARY metric (instead of conversion) and `region` as the factor under study (instead of campaign group). Include: exploration (a plot), a significance test (ANOVA from Module 10), a check of group comparability on spend, a regression model controlling for spend, and a short final report with a recommendation (e.g.: "is it worth designing regionally differentiated campaigns?").

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
from statsmodels.formula.api import ols
import statsmodels.api as sm

# 1. exploration
customers.boxplot(column="satisfaction", by="region")
plt.suptitle("")
plt.show()

# 2. significance test (ANOVA)
region_groups = [data["satisfaction"].values for _, data in customers.groupby("region")]
f_stat, p_anova = stats.f_oneway(*region_groups)
print(f"ANOVA: F={f_stat:.3f}, p={p_anova:.6f}")

# 3. group comparability on spend
print(customers.groupby("region")["monthly_spend"].mean())

# 4. model controlling for spend
region_model = ols("satisfaction ~ C(region) + monthly_spend", data=customers).fit()
print(region_model.summary().tables[1])
```

Unlike the campaign_group/satisfaction case from section 5, here the region effect REMAINS significant even after controlling for spend -- because back in Module 8 we deliberately built a real, systematic region effect on satisfaction (region_effect), unlike the purely coincidental confound from section 5. This is a good illustration of the difference: controlling for variables removes APPARENT effects, but it doesn't remove REAL ones.
</details>

## Congratulations!

This is the end of the Statistics for Data Analysts/Engineers course. Across 15 modules
we went from the basics of descriptive statistics, through probability and sampling,
hypothesis testing, t-tests/chi-square/ANOVA, correlation and regression (linear and
logistic), all the way to designing A/B tests and a complete end-to-end analysis. The
most important skill we hope you take away from this isn't memorizing specific formulas
or `scipy`/`statsmodels` functions -- it's **thinking critically about data**: asking "is
this really significant, or just noise?", "is this really causal, or just correlation?",
and "do I have enough evidence to actually recommend this?".